# Interpreting and Validating FHIR Resources

## Objectives
After completing this lab, you will be able to:
- Interpret the structure of common FHIR resources.  
- Navigate nested elements within FHIR JSON documents.  
- Validate the presence of required FHIR fields.  
- Identify interoperability issues caused by incomplete or inconsistent data.


## What you will do in this lab
In this lab, you'll work with sample FHIR JSON resources to understand how clinical data is structured and exchanged across healthcare systems.


You will:
- Load and inspect sample FHIR JSON resources.  
- Explore Patient, Encounter, and Observation resources.  
- Examine nested structures and references.  
- Validate required elements in each resource.  
- Identify common interoperability issues.


## Overview
FHIR (Fast Healthcare Interoperability Resources) is a widely adopted standard for exchanging healthcare data.
FHIR resources are represented as structured JSON (JavaScript Object Notation) documents that often contain deeply nested fields and references.
While FHIR provides a standard structure, real-world implementations frequently contain missing or inconsistent data.
This lab focuses on understanding and validating FHIR resources to ensure they can be safely used for analytics and interoperability.


## About the dataset/environment
You will work with **synthetic FHIR JSON resources** that resemble real-world FHIR data.
The lab includes:
- Patient resources  
- Encounter resources  
- Observation resources  

Some required fields are intentionally missing or inconsistent to simulate real interoperability challenges.


## Setup

In [1]:
# This cell loads synthetic FHIR resources from a JSON file.
# It keeps the lab self-contained while separating data from code.

import json
import requests
from pprint import pprint

# URL to the JSON file
json_file_path = "https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab2/fhir_resources.json"

# Fetch JSON from URL
response = requests.get(json_file_path)
response.raise_for_status()  # fails fast if URL is unreachable

# Load JSON content
fhir_resources = response.json()

# Display all FHIR resources
pprint(fhir_resources)


[{'gender': 'male',
  'id': 'patient-001',
  'name': [{'family': 'Doe', 'given': ['John']}],
  'resourceType': 'Patient'},
 {'id': 'encounter-001',
  'period': {'start': '2023-01-05'},
  'resourceType': 'Encounter',
  'status': 'finished',
  'subject': {'reference': 'Patient/patient-001'}},
 {'code': {'text': 'Heart Rate'},
  'id': 'observation-001',
  'resourceType': 'Observation',
  'status': 'final',
  'subject': {'reference': 'Patient/patient-001'},
  'valueQuantity': {'unit': 'beats/minute', 'value': 200}}]


## Step 1: Inspect FHIR resource types
In this step, you will identify the type of each FHIR resource and understand its overall purpose.
This helps you set expectations about which fields should be present.

**Why this matters in healthcare:** Misidentifying a resource type can lead to incorrect assumptions about required data.


In [2]:

# This cell prints the resourceType of each FHIR resource.
# Understanding resource types is the foundation of FHIR interpretation.

for resource in fhir_resources:
    print(resource.get("resourceType"))


Patient
Encounter
Observation


In [3]:
# Get all the resources from the fhir_resources list
patient_resource = fhir_resources[0]
encounter_resource = fhir_resources[1]
observation_resource = fhir_resources[2]

In [4]:
#pretty print all the resources
pprint(patient_resource)
pprint(encounter_resource)
pprint(observation_resource)

{'gender': 'male',
 'id': 'patient-001',
 'name': [{'family': 'Doe', 'given': ['John']}],
 'resourceType': 'Patient'}
{'id': 'encounter-001',
 'period': {'start': '2023-01-05'},
 'resourceType': 'Encounter',
 'status': 'finished',
 'subject': {'reference': 'Patient/patient-001'}}
{'code': {'text': 'Heart Rate'},
 'id': 'observation-001',
 'resourceType': 'Observation',
 'status': 'final',
 'subject': {'reference': 'Patient/patient-001'},
 'valueQuantity': {'unit': 'beats/minute', 'value': 200}}


## Step 2: Explore nested FHIR structures
FHIR resources often contain deeply nested fields.
You will explore how data such as names and codes are stored within nested objects and arrays.

**Why this matters in healthcare:** Incorrect navigation of nested fields can result in missing or misread clinical information.


In [5]:
# This cell explores nested fields in the Patient resource.
# Nested structures are common in FHIR JSON documents.

# Access patient name details safely
patient_name = patient_resource.get("name", [{}])[0]
pprint(patient_name)


{'family': 'Doe', 'given': ['John']}


## Step 3: Validate required elements in Patient
In this step, you will check whether required Patient elements are present.
FHIR defines certain fields that should exist for safe interoperability.

**Why this matters in healthcare:** Missing required fields can prevent systems from correctly identifying or matching patients.


In [6]:

# This cell checks for required fields in the Patient resource.
# Validation helps detect incomplete data early.

required_patient_fields = ["id", "name", "gender", "birthDate"]

for field in required_patient_fields:
    print(field, "present:", field in patient_resource)


id present: True
name present: True
gender present: True
birthDate present: False


## Step 4: Validate Encounter period information
Encounters should have a clear start and end.
You will examine whether Encounter timing fields are complete.

**Why this matters in healthcare:** Incomplete encounter periods can break length-of-stay calculations and reporting.


In [7]:

# This cell checks required timing fields in the Encounter resource.
# Encounter periods are essential for operational and clinical analysis.

encounter_period = encounter_resource.get("period", {})

print("Start present:", "start" in encounter_period)
print("End present:", "end" in encounter_period)


Start present: True
End present: False


## Step 5: Interpret Observation values
Observations contain clinical measurements.
In this step, you will inspect observation values and assess whether they are reasonable.

**Why this matters in healthcare:** Implausible observation values can mislead clinical decisions and analytics.


In [8]:

# This cell examines the Observation valueQuantity.
# Clinical validation often starts with simple range checks.

value = observation_resource.get("valueQuantity", {}).get("value")
unit = observation_resource.get("valueQuantity", {}).get("unit")

print("Observed value:", value)
print("Unit:", unit)


Observed value: 200
Unit: beats/minute


## Step 6: Identify interoperability issues
Finally, you will summarize missing or inconsistent fields across resources.
This helps highlight common interoperability challenges.

**Why this matters in healthcare:** Interoperability failures often stem from small but critical data gaps.


In [9]:

# This cell summarizes detected interoperability issues.
# Identifying issues early helps prevent downstream system failures.

issues = []

if "birthDate" not in patient_resource:
    issues.append("Patient.birthDate missing")

if "end" not in encounter_resource.get("period", {}):
    issues.append("Encounter.period.end missing")

if observation_resource.get("valueQuantity", {}).get("value", 0) > 180:
    issues.append("Observation value out of expected range")

pprint(issues)


['Patient.birthDate missing',
 'Encounter.period.end missing',
 'Observation value out of expected range']


## Exercises

In [10]:
import requests

# URL to the JSON file
json_file_path = "https://fundamentals-of-healthcare-data-science-858397.gitlab.io/labs/lab2/fhir_resources_exercises.json"

# Fetch JSON from URL
response = requests.get(json_file_path)
response.raise_for_status()

# Load JSON content
fhir_resources = response.json()

### Exercise 1: List FHIR resource types

In [11]:
# your code goes here

for r in fhir_resources:
    print(r.get('resourceType'))

Patient
Encounter
Observation
Observation


### Exercise 2: Explore Patient name structure

In [12]:
# your code goes here

patient_resource = fhir_resources[0]
pprint(patient_resource.get('name'))


[{'family': 'Sharma', 'given': ['Anita']}]


### Exercise 3: Check required Patient fields

In [13]:
# your code goes here

for f in ['id', 'name', 'gender', 'birthDate']:
    print(f, f in patient_resource)

id True
name True
gender True
birthDate True


### Exercise 4: Validate Encounter period

In [14]:
# your code goes here

encounter_resource = fhir_resources[1]
print(encounter_resource.get('period'))

{'start': '2024-03-18'}


### Exercise 5: Inspect Observation value

In [15]:
# your code goes here

print(observation_resource.get('valueQuantity'))

{'value': 200, 'unit': 'beats/minute'}
